# Build a knowledge graph from your documents with AutoGraph

In this tutorial, we go from a folder of raw documents to a **queryable knowledge
graph** using **AutoGraph** on the Arango Contextual Data Platform, driving every
step from this notebook. It is the same workflow as the AutoGraph web interface,
but here every step is a Python call against the platform's REST API.

**AutoGraph** imports your files, builds a **Corpus Graph** (embeddings,
similarity edges, and domain clusters), assigns each domain a retrieval strategy
(**VectorRAG** or **FullGraphRAG**), and orchestrates the importers that build the
knowledge graph. You then deploy a **Retriever** and ask questions in natural
language.

**By the end, you will have:**

- An AutoGraph service deployed on the platform, configured with your LLM.
- Your documents uploaded and embedded into a Corpus Graph.
- A per-domain RAG strategy chosen automatically for your content.
- A knowledge graph built by the orchestrated importers.
- A running Retriever service you can query.
- A simple way to remove everything when you are done.

Run the cells **top to bottom**, one at a time - do **not** use **Run All**. The
corpus build, the RAG Strategizer, and the orchestration run in the background,
so you re-run each status cell until it reports the stage is finished before
continuing. The access token is short-lived; if a step fails with an
authentication error, re-run the nearest `authenticate(...)` cell and continue.

## Step 1: Configure access

The notebook reads its configuration from a file, so this is the only place you
edit - you never change code cells to point it at your platform. Create a file
named `env` next to this notebook and fill in your own values:

```sh
SERVER_URL = "https://<EXTERNAL_ENDPOINT>:8529"
USERNAME = "root"
PASSWORD = "<your-password>"
DB_NAME = "your-database"
PROJECT_NAME = "your-autograph-project"
LLM_API_KEY = "sk-..."
FILES_PATH = "./files"
```

`PROJECT_NAME` becomes the prefix for every collection AutoGraph creates.
`FILES_PATH` is the folder of documents to ingest (top-level files only). Then
run the two cells below to install `python-dotenv` and load the file - every
later cell reads these values through `os.environ`.

In [ ]:
%pip install python-dotenv==1.2.1

In [ ]:
import requests
import os
import json
from pprint import pprint
from typing import Dict, Optional
from dotenv import load_dotenv

# Create a env file with the following variables before running this notebook:
# SERVER_URL = "https://your-server"
# USERNAME = "server-username"
# PASSWORD = "server-password"
# DB_NAME = "your-database"
# PROJECT_NAME = "your-project"
# LLM_API_KEY = "sk-p...."
# FILES_PATH = "your-files-path"

load_dotenv(dotenv_path="./env", override=True)  # loads from env file in the working directory

## Step 2: Authenticate

Every AutoGraph API call needs a bearer token from the platform's `/_open/auth`
endpoint. The first cell defines the helper functions used throughout
(`authenticate`, `send_request`, `start_service`, `stop_service`); the second
obtains and stores the token.

In [ ]:
SERVER_URL = os.environ["SERVER_URL"]
VERIFY_TLS = False
jwt_token = None

def authenticate(username: str, password: str) -> str:
    """Authenticate and store a JWT for later requests."""
    global jwt_token
    response = requests.post(
        f"{SERVER_URL}/_open/auth",
        json={"username": username, "password": password},
        verify=VERIFY_TLS,
    )
    response.raise_for_status()
    jwt_token = response.json().get("jwt")
    if not jwt_token:
        raise ValueError("Authentication response did not include a JWT.")
    print("Authenticated.")
    return jwt_token


def send_request(suffix: str, payload: Optional[Dict] = None, method: str = "POST"):
    """Call a GenAI platform endpoint with the JWT."""
    if not jwt_token:
        raise ValueError("Authenticate first.")
    response = requests.request(
        method,
        f"{SERVER_URL}{suffix}",
        json=payload,
        headers={
            "Authorization": f"Bearer {jwt_token}",
            "Content-Type": "application/json",
        },
        verify=VERIFY_TLS,
    )
    response.raise_for_status()
    return response.json() if response.content else {}


def start_service(service_name: str, startup_parameters: Dict):
    return send_request(
        "/gen-ai/v1/service",
        {"service_name": service_name, "env": startup_parameters},
    )


def stop_service(service_id: str):
    return send_request(f"/gen-ai/v1/service/{service_id}", method="DELETE")

In [ ]:
authenticate(os.environ["USERNAME"], os.environ["PASSWORD"])

## Step 3: Create the project

Create a GenAI project. It isolates your data, and its name prefixes every
collection AutoGraph creates. The cell is safe to re-run: if the project already
exists, it catches the error and continues.

In [ ]:
DB_PROJECT_NAME = os.environ["PROJECT_NAME"]
DB_NAME = os.environ["DB_NAME"]

project_payload = {
    "project_name": DB_PROJECT_NAME,
    "project_db_name": DB_NAME,
    "project_type": "autograph",
    "project_description": "Autograph_DEMO",
}

print(f"Creating project '{DB_PROJECT_NAME}' in database '{DB_NAME}'...")
try:
    project_response = send_request("/gen-ai/v1/project", project_payload, "POST")
    print(f"Project '{DB_PROJECT_NAME}' created successfully!")
    print(json.dumps(project_response, indent=2))
except requests.exceptions.HTTPError as e:
    status = e.response.status_code if e.response is not None else None
    body = (e.response.text or "").lower() if e.response is not None else ""
    if status == 409 or (
        status == 400
        and any(
            phrase in body
            for phrase in ("already exists", "already exist", "conflict", "duplicate")
        )
    ):
        print(f"Project '{DB_PROJECT_NAME}' already exists. Continuing...")
    else:
        print(f"Failed to create project '{DB_PROJECT_NAME}' (HTTP {status}):")
        if e.response is not None:
            print(e.response.text)
        raise
except requests.exceptions.RequestException as e:
    print(f"Failed to create project '{DB_PROJECT_NAME}': {e}")
    raise

## Step 4: Store your API key

Store the LLM key once in the Secrets Manager and reference it by `profileId` for
both chat and embeddings, instead of passing raw keys around. The cell prints the
profile ID, which the deployment steps reuse.

In [ ]:
secret_resp = send_request(
    "/gen-ai/v1/secrets",
    {
        "profile_type": "LLM",
        "name": "API_KEY",
        "secret_data": os.environ["LLM_API_KEY"],
        "description": "Autograph_DEMO",
        "provider": "openai",
    },
)
profile_id = secret_resp["profile"]["profileId"]
print(profile_id)

## Step 5: Deploy the AutoGraph service

Deploy the AutoGraph service as a pod, using your models and the secret profile
from Step 4. Fill in `myDict`, then run the cells below to start the service,
capture its ID, build the pod-scoped request helper, and confirm it is healthy
(a healthy service returns `{"status": "SERVING"}`). If the health check fails,
the service is still starting - wait a few seconds and re-run it.

In [ ]:
%%time
# Suppress SSL warnings
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
# Start the AutoGraph Importer service

authenticate(os.environ["USERNAME"], os.environ["PASSWORD"])

myDict = {
    # --- Required ---
    "db_name": os.environ["DB_NAME"],
    "chat_api_provider": "openai",
    "embedding_api_provider": "openai",
    "chat_model": "gpt-5.4-nano",
    "embedding_model": "text-embedding-3-small",
    "chat_secret_profile_id": profile_id,        # TODO: Secret Manager profileId for chat LLM
    "embedding_secret_profile_id": profile_id,   # TODO: Secret Manager profileId for embeddings
    "genai_project_name": os.environ["PROJECT_NAME"],
    "chat_api_url": "https://api.openai.com/v1",
    "embedding_api_url": "https://api.openai.com/v1",

    # --- Optional: pod resource allocation ---
    "profiles": "",                           # Kubernetes resource profile(s) for the pod (e.g. "memory-16gi-cpu-2"); comma-separated for multiple
}

response = start_service("arangodb-autograph", myDict)
pprint(response)

In [ ]:
autograph_service_id = response["serviceInfo"]["serviceId"].split("-")[-1]
autograph_service_id

In [ ]:
AG_BASE = f"{SERVER_URL}/autograph/{autograph_service_id}"

SESSION = requests.Session()
SESSION.headers.update({
    "Content-Type": "application/json",
    "Accept": "application/json",
})
TIMEOUT_SECS = 60


def ag_request(method: str, path: str, payload: dict | None = None, query: dict | None = None):
    """Call an AutoGraph pod endpoint."""
    if not jwt_token:
        raise ValueError("Authenticate first.")
    params = {"token": jwt_token}
    if query:
        params.update(query)
    resp = SESSION.request(
        method=method,
        url=f"{AG_BASE}{path}",
        json=payload,
        params=params,
        headers={"Authorization": f"Bearer {jwt_token}"},
        timeout=TIMEOUT_SECS,
        verify=VERIFY_TLS,
        allow_redirects=False,
    )
    resp.raise_for_status()
    return resp.json() if resp.content else {}


print(f"AG_BASE = {AG_BASE}")

In [ ]:
# Check if AutoGraph service is healthy and ready
authenticate(os.environ["USERNAME"], os.environ["PASSWORD"])
try:
    health_response = ag_request("GET", "/v1/health")
    print("AutoGraph service is healthy!")
    pprint(health_response)
except Exception as e:
    print(f"AutoGraph service health check failed: {e}")
    print("Please wait a few moments for the service to fully start, then retry.")

## Step 6: Upload your documents

Upload every top-level file in `FILES_PATH` to the File Manager, a separate
platform service. The cell prints each file as it uploads and collects the
returned IDs into `rag_uploaded_file_ids`, which the corpus build uses next.

In [ ]:
# Upload all files in FM_FOLDER to File Manager; fills rag_uploaded_file_ids for corpus build.
from pathlib import Path

FM_DATABASE = os.environ["DB_NAME"]  # same as db_name in start_service
FM_FOLDER = os.environ["FILES_PATH"]  # path relative to cwd, or e.g. Path("/home/jovyan/Modules/Office")

if not jwt_token:
    raise ValueError("Run authenticate() first.")

verify_tls = VERIFY_TLS if "VERIFY_TLS" in globals() else os.getenv("VERIFY_TLS", "true").lower() == "true"
api_root = f"{SERVER_URL.rstrip('/')}/_platform/filemanager"
folder = Path(FM_FOLDER).expanduser().resolve()
if not folder.is_dir():
    raise NotADirectoryError(folder)

files = sorted(
    [p for p in folder.iterdir() if p.is_file() and not p.name.startswith(".")],
    key=lambda p: p.name.lower(),
)
if not files:
    raise FileNotFoundError(f"No files in {folder}")

rag_uploaded_file_ids = []
for i, path in enumerate(files, 1):
    print(f"{i}/{len(files)} {path.name}")
    url = f"{api_root}/_db/{FM_DATABASE}/rag-input"
    with open(path, "rb") as fh:
        r = requests.post(
            url,
            headers={"Authorization": f"Bearer {jwt_token}"},
            data={"name": path.name},
            files={"file": (path.name, fh)},
            timeout=300,
            verify=verify_tls,
        )
    r.raise_for_status()
    rag_uploaded_file_ids.append(r.json()["id"])

pprint(rag_uploaded_file_ids)


## Step 7: Build the corpus

Start an asynchronous corpus build from the uploaded files. It embeds each
document, finds similarity relationships (vector plus lexical search fused with
Reciprocal Rank Fusion), and clusters documents into domains with the Leiden
algorithm. The first cell starts the build; the second polls its status - re-run
it every 10-30 seconds until `status` is `completed` before continuing.

In [ ]:
# --- Corpus Build ---
# Requires rag_uploaded_file_ids from the File Manager upload cell.

build_payload = {
    "embedding_strategy": "first_chunk",
    "file_ids": rag_uploaded_file_ids,
    "strategy": {
        "top_k": 7,
        "cluster_threshold": 1,
    },
}
build_corpus_graph = ag_request("POST", "/v1/corpus/builds", payload=build_payload)
pprint(build_corpus_graph)

In [ ]:
# Extract the corpus_build_id from the build response
corpus_build_id = build_corpus_graph["corpusBuildId"]

# Check build status
build_status = ag_request(
    "GET",                                    # method
    f"/v1/corpus/builds/{corpus_build_id}",  # path with build ID
)
pprint(build_status)

## Step 8: Generate strategies

Run the RAG Strategizer. For each domain it scores complexity, extracts entity
types, and assigns **VectorRAG** or **FullGraphRAG** (controlled by
`fullGraphRagStrategy`), writing the results to the `rags` collection. Run
`analyze`, wait for the background job to finish, then read the strategies with
the GET cell - re-run it until the rows appear.

In [ ]:
authenticate(os.environ["USERNAME"], os.environ["PASSWORD"])

In [ ]:
rag_payload = {
    "fullGraphRagStrategy": "very high", #TODO add value 
}

rag_response = ag_request(
    "POST",
    "/v1/rag-strategizer/analyze",
    payload=rag_payload,
)

from pprint import pprint
pprint(rag_response)


In [ ]:
# GET endpoint to retrieve all RAG strategies
rag_response = ag_request(
    "GET", "/v1/rag-strategizer/strategy",
)

pprint(rag_response)

## Step 9: Import into the knowledge graph

Orchestration spawns GraphRAG importer worker pods, loads the jobs the
strategizer wrote to `rags`, and builds the knowledge graph for every partition.
It returns an `orchestration_id` immediately while the import runs in the
background. Only one orchestration run should be active at a time.

In [ ]:
orchestrate_payload = {
    "chat_secret_profile_ids": [profile_id], #TODO add profileId
    "embedding_secret_profile_id": profile_id, #TODO add profileId
    "max_retries": 1,
    "replicas": 2,
    # "partition_ids": ["...", "...", "..."] #TODO add partition Id
}

orchestrate_response = ag_request(
    "POST",
    "/v1/orchestrate",
    payload=orchestrate_payload,
)

from pprint import pprint
pprint(orchestrate_response)

## Step 10: Track progress (optional)

While the async stages run, this optional cell calls the project metadata API to
report the status of the corpus build, strategizer, and importer orchestration in
one place. It is not required for the main flow.

In [ ]:
#get metadata
import requests
import json
import urllib3
import os

# Disable SSL warnings if using self-signed certificates
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Configuration
base_url = os.environ["SERVER_URL"]
project_db_name = os.environ["DB_NAME"] #TODO add db name
project_name = os.environ["PROJECT_NAME"] #TODO add project name
VERIFY_TLS = False

# Independent JWT fetch (no manual token paste)
username = os.getenv("USERNAME", "root")
password = os.getenv("PASSWORD")
if not password:
    raise ValueError("Set PASSWORD env var before running this cell.")

auth_url = f"{base_url}/_open/auth"
auth_response = requests.post(
    auth_url,
    json={"username": username, "password": password},
    verify=VERIFY_TLS,
)
auth_response.raise_for_status()
bearer_token = auth_response.json().get("jwt")
if not bearer_token:
    raise ValueError("Authentication succeeded but no jwt returned.")

# Construct the endpoint URL with /gen-ai/v1 prefix
url = f"{base_url}/gen-ai/v1/project_by_name/{project_db_name}/{project_name}"

# Set up headers
headers = {
    "Authorization": f"Bearer {bearer_token}",
    "Content-Type": "application/json"
}

# Make the GET request
response = requests.get(url, headers=headers, verify=VERIFY_TLS)

# Check response
if response.status_code == 200:
    project_data = response.json()
    print("Project retrieved successfully:")
    print(json.dumps(project_data, indent=2))

    # Access specific fields (note: field names are camelCase in JSON)
    print(f"\nProject Name: {project_data.get('projectName')}")
    print(f"Project Type: {project_data.get('projectType')}")
    print(f"Database: {project_data.get('projectDbName')}")

    # Access metadata
    if 'projectMetadata' in project_data:
        metadata = project_data['projectMetadata']
        print(f"\nImporter Services: {len(metadata.get('importerServices', []))}")
        print(f"Retriever Services: {len(metadata.get('retrieverServices', []))}")

        # AutoGraph service (singular object in API)
        svc = metadata.get('autographService')
        print(f"AutoGraph Services: {1 if isinstance(svc, dict) else 0}")

In [ ]:
authenticate(os.environ["USERNAME"], os.environ["PASSWORD"])

## Step 11: Stop the AutoGraph service

The knowledge graph is now built and persists in ArangoDB, so you can stop the
AutoGraph service to free cluster resources. Stopping the service does not delete
your data; the Retriever you deploy next is a separate service.

In [ ]:
# NOTE: Remember to stop the service when you are done to free up resources!

response = stop_service(f"arangodb-autograph-{autograph_service_id}")

print(response)

## Step 12: Deploy the Retriever

Deploy the Retriever service to query your knowledge graph. It starts the same
way as AutoGraph, with the same LLM configuration. Run the cells below to deploy
it, capture its ID, and confirm it is healthy. The service can report healthy a
moment before it is fully ready; if a query fails right after deployment, wait a
few seconds and retry.

In [ ]:
%%time
# Suppress SSL warnings
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
# Start the AutoGraph Importer service

authenticate(os.environ["USERNAME"], os.environ["PASSWORD"])

myDict = {
    # --- Required ---
    "db_name": os.environ["DB_NAME"],
    "chat_api_provider": "openai",
    "embedding_api_provider": "openai",
    "chat_model": "gpt-5.4-nano",
    "embedding_model": "text-embedding-3-small",
    "chat_secret_profile_id": profile_id,        # TODO: Secret Manager profileId for chat LLM
    "embedding_secret_profile_id": profile_id,   # TODO: Secret Manager profileId for embeddings
    "genai_project_name": os.environ["PROJECT_NAME"],
    "chat_api_url": "https://api.openai.com/v1",
    "embedding_api_url": "https://api.openai.com/v1",

    # --- Optional: pod resource allocation ---
    "profiles": "",                           # Kubernetes resource profile(s) for the pod (e.g. "memory-16gi-cpu-2"); comma-separated for multiple
}

rertiever_response = start_service("arangodb-graphrag-retriever", myDict)
pprint(rertiever_response)

In [ ]:
retriever_service_id = rertiever_response["serviceInfo"]["serviceId"].split("-")[-1]
retriever_service_id

In [ ]:
# Check if Retriever service is healthy and ready
authenticate(os.environ["USERNAME"], os.environ["PASSWORD"])
try:
    health_response = send_request(
        f"/graphrag/retriever/{retriever_service_id}/v1/health",
        method="GET",
    )
    print("Retriever service is healthy!")
    pprint(health_response)
except Exception as e:
    print(f"Retriever service health check failed: {e}")
    print("Please wait a few moments for the service to fully start, then retry.")

## Step 13: Query the knowledge graph

Ask questions against the knowledge graph you built. Each cell below uses a
different search mode - run whichever fits your question:

| Query type | `query_type` | Best for |
|---|---|---|
| **Global** | `1` | Broad themes and summaries across the whole graph |
| **Local** | `2` | Specific entities, relationships, and details |
| **Unified** | `3` | Combined chunk + entity search for a single answer |
| **Deep** | `2` with `use_llm_planner: true` | Complex, multi-step, LLM-planned questions |

This is the payoff: you are now asking your own documents questions and getting
answers back from the knowledge graph.

In [ ]:
%%time
# Global Search (query_type=1)
myBody = {
    "query": "What are the dominant themes across the imported documentation regarding knowledge graphs, GenAI, and enterprise data strategy?",
    "query_type": 1,
    # "include_metadata": True,
    # "use_cache": True,
}

retrieverResponse = send_request(f"/graphrag/retriever/{retriever_service_id}/v1/graphrag-query", myBody, "POST")

pprint(retrieverResponse)

In [ ]:
%%time
# Local Search (query_type=2)
myBody = {
    "query": "What specific claims does the content make when comparing ArangoDB to Neo4j, including any cited metrics or workloads?",
    "query_type": 2,
    # "include_metadata": True,
    # "use_cache": True,
    "use_llm_planner": False,
}

retrieverResponse = send_request(f"/graphrag/retriever/{retriever_service_id}/v1/graphrag-query", myBody, "POST")

pprint(retrieverResponse)

In [ ]:
%%time
# Unified Search (query_type=3)
myBody = {
    "query": "Explain how vector search is presented in ArangoDB’s documentation, combining the main technical ideas with representative examples or details from the text.",
    "query_type": 3,
    # "include_metadata": True,
    # "use_cache": True,
}

retrieverResponse = send_request(f"/graphrag/retriever/{retriever_service_id}/v1/graphrag-query", myBody, "POST")

pprint(retrieverResponse)

In [ ]:
%%time
# Deep Search (query_type=2, by defult use_llm_planner=true)
myBody = {
    "query": "Identify the corpus’s central arguments for why AI fails without sufficient context; then map each argument to the ArangoDB or graph-related capabilities the documents propose as remedies; note any inconsistencies across page types.",
    "query_type": 2,
    # "include_metadata": True,
    # "use_cache": True,
}

retrieverResponse = send_request(f"/graphrag/retriever/{retriever_service_id}/v1/graphrag-query", myBody, "POST")

pprint(retrieverResponse)

## Step 14: Clean up

Stop the Retriever service when you are finished to free resources. Your ArangoDB
data - collections, graphs, and documents - persists after the pods are stopped,
so you can redeploy a service later and keep querying.

In [ ]:
# NOTE: Remember to stop the service when you are done to free up resources!

response = stop_service(f"arangodb-graphrag-retriever-{retriever_service_id}")

print(response)